# Elite Competition Solution

In [ ]:

import pandas as pd
import numpy as np

from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from scipy.stats import rankdata

train = pd.read_csv('input/train.csv')
test = pd.read_csv('input/test.csv')
sample = pd.read_csv('input/sample_submission.csv')

TARGET = 'Drafted'
ID_COL = 'Id'

y = train[TARGET]

train_features = train.drop(columns=[TARGET])

full = pd.concat([train_features, test], axis=0).reset_index(drop=True)

cat_cols = [c for c in full.columns if full[c].dtype == 'object']

for col in cat_cols:
    le = LabelEncoder()
    full[col] = le.fit_transform(full[col].astype(str))

# Feature engineering

if {'Vertical_Jump', 'Broad_Jump'}.issubset(full.columns):
    full['Explosion_Index'] = full['Vertical_Jump'] * full['Broad_Jump']

if {'Weight', 'Sprint_40yd'}.issubset(full.columns):
    full['Speed_Power'] = full['Weight'] / (full['Sprint_40yd'] + 1e-6)

if {'BMI', 'Sprint_40yd'}.issubset(full.columns):
    full['BMI_Speed'] = full['BMI'] / (full['Sprint_40yd'] + 1e-6)

if 'Position' in full.columns:

    numeric_cols = full.select_dtypes(include=np.number).columns

    for col in numeric_cols:

        mean = full.groupby('Position')[col].transform('mean')
        std = full.groupby('Position')[col].transform('std')

        full[f'{col}_z'] = (
            (full[col] - mean) / (std + 1e-6)
        )

train_processed = full.iloc[:len(train)].copy()
test_processed = full.iloc[len(train):].copy()

cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=3,
    random_state=42
)

target_cols = [
    c for c in ['School', 'Position', 'Player_Type']
    if c in train.columns
]

for col in target_cols:

    train_encoded = np.zeros(len(train_processed))
    test_encoded = np.zeros(len(test_processed))

    for tr_idx, va_idx in cv.split(train_processed, y):

        means = (
            train.iloc[tr_idx]
            .groupby(col)[TARGET]
            .mean()
        )

        train_encoded[va_idx] = (
            train.iloc[va_idx][col]
            .map(means)
            .fillna(y.mean())
        )

    global_means = train.groupby(col)[TARGET].mean()

    test_encoded = (
        test[col]
        .map(global_means)
        .fillna(y.mean())
    )

    train_processed[f'{col}_TE'] = train_encoded
    test_processed[f'{col}_TE'] = test_encoded

features = [
    c for c in train_processed.columns
    if c != ID_COL
]

X = train_processed[features]
X_test = test_processed[features]

seeds = [42, 100, 777, 2024]

pred_cat = np.zeros(len(X_test))
pred_lgb = np.zeros(len(X_test))
pred_xgb = np.zeros(len(X_test))

for seed in seeds:

    for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y)):

        X_train = X.iloc[tr_idx]
        X_valid = X.iloc[va_idx]

        y_train = y.iloc[tr_idx]
        y_valid = y.iloc[va_idx]

        cat_model = CatBoostClassifier(
            iterations=6000,
            learning_rate=0.01,
            depth=10,
            eval_metric='AUC',
            random_seed=seed,
            verbose=0
        )

        cat_model.fit(
            X_train,
            y_train,
            eval_set=(X_valid, y_valid),
            use_best_model=True
        )

        pred_cat += (
            cat_model.predict_proba(X_test)[:, 1]
            / (len(seeds) * cv.get_n_splits())
        )

        lgb_model = LGBMClassifier(
            n_estimators=4000,
            learning_rate=0.01,
            num_leaves=128,
            random_state=seed
        )

        lgb_model.fit(X_train, y_train)

        pred_lgb += (
            lgb_model.predict_proba(X_test)[:, 1]
            / (len(seeds) * cv.get_n_splits())
        )

        xgb_model = XGBClassifier(
            n_estimators=4000,
            learning_rate=0.01,
            max_depth=8,
            eval_metric='logloss',
            random_state=seed
        )

        xgb_model.fit(X_train, y_train)

        pred_xgb += (
            xgb_model.predict_proba(X_test)[:, 1]
            / (len(seeds) * cv.get_n_splits())
        )

rank_cat = rankdata(pred_cat)
rank_lgb = rankdata(pred_lgb)
rank_xgb = rankdata(pred_xgb)

final_predictions = (
    0.6 * rank_cat +
    0.25 * rank_lgb +
    0.15 * rank_xgb
)

final_predictions = final_predictions / final_predictions.max()

submission = pd.DataFrame({
    'Id': test[ID_COL],
    'Drafted': final_predictions
})

submission.to_csv('submission.csv', index=False)

print(submission.head())
print(submission.shape)
